# Phase-1 Project Setup

In [1]:


import os
import warnings

import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 60)
print("PDF Chatbot Project Setup")
print("=" * 60)

PDF Chatbot Project Setup


In [2]:
# ==========================================================
# Check Project Directories
# ==========================================================

folders = [

    "../dataset",

    "../model",

    "../vectorstore",

    "../charts"

]

for folder in folders:

    os.makedirs(folder, exist_ok=True)

print("=" * 60)

print("Project Directories Created Successfully")

print("=" * 60)

for folder in folders:

    print(folder)

Project Directories Created Successfully
../dataset
../model
../vectorstore
../charts


In [3]:
# ==========================================================
# Load Environment Variables
# ==========================================================

import os
import subprocess
import sys

try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv"])
    from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "").strip()

if GOOGLE_API_KEY:
    if len(GOOGLE_API_KEY) >= 10 and GOOGLE_API_KEY.count(".") >= 1:
        print("=" * 60)
        print("Gemini API Key Loaded Successfully")
        print("=" * 60)
    else:
        print("=" * 60)
        print("ERROR : GOOGLE_API_KEY format looks invalid")
        print("=" * 60)
else:
    print("=" * 60)
    print("ERROR : GOOGLE_API_KEY Not Found")
    print("=" * 60)


Gemini API Key Loaded Successfully


In [4]:
# ==========================================================
# Verify Libraries
# ==========================================================

import streamlit
import langchain
import faiss
import fitz
import google.generativeai

print("=" * 60)

print("Library Verification Successful")

print("=" * 60)

print("Streamlit :", streamlit.__version__)
print("LangChain :", langchain.__version__)
print("FAISS Imported Successfully")
print("PyMuPDF Imported Successfully")
print("Google Generative AI Imported Successfully")

Library Verification Successful
Streamlit : 1.60.0
LangChain : 1.3.14
FAISS Imported Successfully
PyMuPDF Imported Successfully
Google Generative AI Imported Successfully


## PHASE 2 : PDF LOADING & TEXT EXTRACTION

In [5]:
# ==========================================================
# PHASE 2 : LOAD PDF FILES
# ==========================================================

import os
from pathlib import Path

# Dataset Folder
DATASET_PATH = "../dataset"

pdf_files = list(Path(DATASET_PATH).glob("*.pdf"))

print("=" * 60)
print("PDF Files Found")
print("=" * 60)

if len(pdf_files) == 0:
    print("No PDF files found.")
else:
    for i, pdf in enumerate(pdf_files, start=1):
        print(f"{i}. {pdf.name}")

print("\nTotal PDFs :", len(pdf_files))

PDF Files Found
1. CN (4).pdf

Total PDFs : 1


In [6]:
# ==========================================================
# Extract Text from PDFs
# ==========================================================

import fitz  # PyMuPDF

documents = []

for pdf_path in pdf_files:

    doc = fitz.open(pdf_path)

    for page_number in range(len(doc)):

        page = doc.load_page(page_number)

        text = page.get_text()

        documents.append({

            "Document": pdf_path.name,

            "Page": page_number + 1,

            "Text": text

        })

    doc.close()

print("=" * 60)
print("PDF Text Extraction Completed")
print("=" * 60)

print("Total Pages Extracted :", len(documents))

PDF Text Extraction Completed
Total Pages Extracted : 46


In [7]:
# ==========================================================
# Create DataFrame
# ==========================================================

import pandas as pd

df = pd.DataFrame(documents)

print("=" * 60)
print("Dataset Created")
print("=" * 60)

print("Shape :", df.shape)

display(df.head())

# ----------------------------------------------------------
# Text Statistics
# ----------------------------------------------------------

df["Characters"] = df["Text"].apply(len)

df["Words"] = df["Text"].apply(lambda x: len(x.split()))

print("\nAverage Characters :", round(df["Characters"].mean(),2))

print("Average Words :", round(df["Words"].mean(),2))

Dataset Created
Shape : (46, 3)


,Document,Page,Text
0,CN (4).pdf,1,COMPUTER NETWORKS LABORATORY \n​\n​\n​\n ​\...
1,CN (4).pdf,2,\nS. No. \nWeek No. \nName of Practical \n1 \...
2,CN (4).pdf,3,device available in the network. \nb) Label e...
3,CN (4).pdf,4,\nc) Use a different topology for each depart...
4,CN (4).pdf,5,DR. B.R. AMBEDKAR NATIONAL \nINSTITUTE OF TECH...



Average Characters : 637.65
Average Words : 89.85


In [8]:
# ==========================================================
# Save Extracted Text
# ==========================================================

os.makedirs("../reports", exist_ok=True)

df.to_csv(

    "../reports/extracted_text.csv",

    index=False

)

print("=" * 60)
print("PHASE 2 COMPLETED")
print("=" * 60)

print(f"""

PDF Files Loaded :

{len(pdf_files)}

Pages Extracted :

{len(df)}

Generated File

✓ extracted_text.csv

Ready For

Phase 3 : Text Cleaning & Chunking

""")

PHASE 2 COMPLETED


PDF Files Loaded :

1

Pages Extracted :

46

Generated File

✓ extracted_text.csv

Ready For

Phase 3 : Text Cleaning & Chunking




# PHASE 3 : TEXT CLEANING & CHUNKING
🎯 Objective

- In this phase, we will:

- Load the extracted text
- Remove empty pages
- Clean unnecessary spaces
- Split the text into chunks
- Add metadata (document name, page number, chunk ID)
- Save the chunks for embedding generation

- Why chunking?
LLMs cannot efficiently process an entire book or large PDF at once. Splitting documents into smaller, meaningful chunks enables accurate retrieval during question answering.

In [9]:
# ==========================================================
# PHASE 3 : TEXT CLEANING & CHUNKING
# ==========================================================

import pandas as pd

DATASET_PATH = "../reports/extracted_text.csv"

df = pd.read_csv(DATASET_PATH)

print("=" * 60)
print("Extracted Text Loaded Successfully")
print("=" * 60)

print("Dataset Shape :", df.shape)

display(df.head())

Extracted Text Loaded Successfully
Dataset Shape : (46, 5)


,Document,Page,Text,Characters,Words
0,CN (4).pdf,1,COMPUTER NETWORKS LABORATORY \n​\n​\n​\n ​\...,361,50
1,CN (4).pdf,2,\nS. No. \nWeek No. \nName of Practical \n1 \...,1410,209
2,CN (4).pdf,3,device available in the network. \nb) Label e...,1477,236
3,CN (4).pdf,4,\nc) Use a different topology for each depart...,417,59
4,CN (4).pdf,5,DR. B.R. AMBEDKAR NATIONAL \nINSTITUTE OF TECH...,185,22


In [10]:
# ==========================================================
# Text Cleaning
# ==========================================================

import re

def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)

    # Remove tabs/newlines
    text = text.replace("\n", " ")

    text = text.replace("\t", " ")

    return text.strip()

df["Clean_Text"] = df["Text"].apply(clean_text)

# Remove Empty Pages
before = len(df)

df = df[df["Clean_Text"] != ""]

after = len(df)

print("=" * 60)
print("Text Cleaning Completed")
print("=" * 60)

print("Pages Before :", before)
print("Pages After  :", after)

Text Cleaning Completed
Pages Before : 46
Pages After  : 44


In [11]:
# ==========================================================
# Document Chunking
# ==========================================================

import pandas as pd
import re

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ModuleNotFoundError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

if "df" not in globals():
    DATASET_PATH = "../reports/extracted_text.csv"
    df = pd.read_csv(DATASET_PATH)

if "Clean_Text" not in df.columns:
    def clean_text(text):
        if pd.isna(text):
            return ""
        text = str(text)
        text = re.sub(r"\s+", " ", text)
        text = text.replace("\n", " ")
        text = text.replace("\t", " ")
        return text.strip()

    df["Clean_Text"] = df["Text"].apply(clean_text)
    df = df[df["Clean_Text"] != ""]

splitter = RecursiveCharacterTextSplitter(

    chunk_size=1000,

    chunk_overlap=200,

    separators=["\n\n", "\n", ".", " ", ""]

)

chunks = []

chunk_id = 1

for _, row in df.iterrows():

    texts = splitter.split_text(row["Clean_Text"])

    for chunk in texts:

        chunks.append({

            "Chunk_ID": chunk_id,

            "Document": row["Document"],

            "Page": row["Page"],

            "Chunk": chunk

        })

        chunk_id += 1

chunk_df = pd.DataFrame(chunks)

print("=" * 60)
print("Chunking Completed")
print("=" * 60)

print("Total Chunks :", len(chunk_df))

display(chunk_df.head())

Chunking Completed
Total Chunks : 54


,Chunk_ID,Document,Page,Chunk
0,1,CN (4).pdf,1,COMPUTER NETWORKS LABORATORY ​ ​ ​ ​ ​ (CSDC -...
1,2,CN (4).pdf,2,S. No. Week No. Name of Practical 1 Week 1 a)​...
2,3,CN (4).pdf,2,. (d)​Implement a unique network topology for ...
3,4,CN (4).pdf,3,device available in the network. b) Label each...
4,5,CN (4).pdf,3,. c) Use a different topology for each departm...


In [12]:
# ==========================================================
# Save Chunked Dataset
# ==========================================================

chunk_df.to_csv(

    "../reports/chunked_documents.csv",

    index=False

)

print("=" * 60)
print("PHASE 3 COMPLETED")
print("=" * 60)

print(f"""

Original Pages

{len(df)}

Generated Chunks

{len(chunk_df)}

Chunk Size

1000 Characters

Chunk Overlap

200 Characters

Saved File

✓ chunked_documents.csv

Ready For

Phase 4 : Embeddings & FAISS Vector Database

""")

PHASE 3 COMPLETED


Original Pages

44

Generated Chunks

54

Chunk Size

1000 Characters

Chunk Overlap

200 Characters

Saved File

✓ chunked_documents.csv

Ready For

Phase 4 : Embeddings & FAISS Vector Database




## PHASE 4 : EMBEDDINGS & FAISS VECTOR DATABASE
- 🎯 Objective

- In this phase, we will:

- Load the chunked documents
- Generate embeddings using a Sentence Transformer
- Build a FAISS vector database
- Save the vector database locally
- Verify semantic search

In [13]:
# ==========================================================
# PHASE 4 : EMBEDDINGS & FAISS VECTOR DATABASE
# ==========================================================

import pandas as pd

DATASET_PATH = "../reports/chunked_documents.csv"

chunk_df = pd.read_csv(DATASET_PATH)

print("=" * 60)
print("Chunked Documents Loaded Successfully")
print("=" * 60)

print("Total Chunks :", len(chunk_df))

display(chunk_df.head())

Chunked Documents Loaded Successfully
Total Chunks : 54


,Chunk_ID,Document,Page,Chunk
0,1,CN (4).pdf,1,COMPUTER NETWORKS LABORATORY ​ ​ ​ ​ ​ (CSDC -...
1,2,CN (4).pdf,2,S. No. Week No. Name of Practical 1 Week 1 a)​...
2,3,CN (4).pdf,2,. (d)​Implement a unique network topology for ...
3,4,CN (4).pdf,3,device available in the network. b) Label each...
4,5,CN (4).pdf,3,. c) Use a different topology for each departm...


In [14]:
# ==========================================================
# Generate Embeddings
# ==========================================================

%pip install -q sentence-transformers langchain-huggingface

import sys

print("Python Executable :", sys.executable)

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ModuleNotFoundError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"

)

print("=" * 60)
print("Embedding Model Loaded Successfully")
print("=" * 60)


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Python Executable : c:\Users\G\AppData\Local\Programs\Python\Python311\python.exe


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2394.55it/s]


Embedding Model Loaded Successfully


In [15]:
# ==========================================================
# Build FAISS Vector Database
# ==========================================================

import os

%pip install -q langchain-community langchain-core faiss-cpu

try:
    from langchain_community.vectorstores import FAISS
except ModuleNotFoundError:
    try:
        from langchain.vectorstores import FAISS
    except ModuleNotFoundError as exc:
        raise ImportError(
            "FAISS import failed. Please make sure langchain-community and faiss-cpu are installed in this notebook kernel."
        ) from exc

from langchain_core.documents import Document

documents = []

for _, row in chunk_df.iterrows():

    documents.append(

        Document(

            page_content=row["Chunk"],

            metadata={

                "document": row["Document"],

                "page": int(row["Page"]),

                "chunk_id": int(row["Chunk_ID"])

            }

        )

    )

try:
    vector_db = FAISS.from_documents(

        documents,

        embedding_model

    )
except Exception as exc:
    raise RuntimeError(f"Failed to create the FAISS index: {exc}") from exc

os.makedirs("../vectorstore", exist_ok=True)

try:
    vector_db.save_local("../vectorstore/faiss_index")
except Exception as exc:
    raise RuntimeError(f"Failed to save the FAISS index: {exc}") from exc

print("=" * 60)
print("FAISS Vector Database Created Successfully")
print("=" * 60)

print("Total Documents :", len(documents))


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
FAISS Vector Database Created Successfully
Total Documents : 54


In [16]:
# ==========================================================
# Test Vector Search
# ==========================================================

vector_db = FAISS.load_local(

    "../vectorstore/faiss_index",

    embedding_model,

    allow_dangerous_deserialization=True

)

query = "What is Machine Learning?"

results = vector_db.similarity_search(

    query,

    k=3

)

print("=" * 60)
print("Top 3 Retrieved Chunks")
print("=" * 60)

for i, result in enumerate(results, start=1):

    print(f"\nResult {i}")

    print("Document :", result.metadata["document"])

    print("Page :", result.metadata["page"])

    print("-" * 50)

    print(result.page_content[:500])

print("\n" + "=" * 60)
print("PHASE 4 COMPLETED")
print("=" * 60)

print("""

Generated

✓ Embeddings

✓ FAISS Vector Database

✓ Semantic Search Test

Saved Folder

✓ vectorstore/faiss_index/

Ready For

Phase 5 : Retrieval-Augmented Generation (RAG)

""")

Top 3 Retrieved Chunks

Result 1
Document : CN (4).pdf
Page : 1
--------------------------------------------------
COMPUTER NETWORKS LABORATORY ​ ​ ​ ​ ​ (CSDC - 0236) B.Tech IV th Semester (January – June 2025) Submitted by :- Khushal Garg (23103077) Group-G3 Submitted to :- Dr. Samayveer Singh Department of Computer Science & Engineering Dr. B. R. Ambedkar National Institute of Technology Jalandhar -144008, Punjab, India

Result 2
Document : CN (4).pdf
Page : 34
--------------------------------------------------
data = list(input("Enter binary data: ")) print("Hamming Code: ", hamming_code(data)) elif choice == '5': break else: print("Invalid choice. Try again.") if __name__ == "__main__": main() Error Detection and Correction Techniques Error detection and correction are essential techniques in computer networks and digital communication to ensure reliable data transmission. Errors can occur due to noise, interference, or hardware faults, leading to corrupted data. To resolve such i

# PHASE 5 : RETRIEVAL-AUGMENTED GENERATION (RAG)
- 🎯 Objective

- In this phase, we will:

- Load the FAISS vector database
- Load the Gemini LLM
- Create the Retriever
- Build the RetrievalQA chain
- Test the chatbot with a sample question

In [17]:
# ==========================================================
# PHASE 5 : RAG PIPELINE
# ==========================================================

import os
from dotenv import load_dotenv

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

embedding_model = HuggingFaceEmbeddings(

    model_name="sentence-transformers/all-MiniLM-L6-v2"

)

vector_db = FAISS.load_local(

    "../vectorstore/faiss_index",

    embedding_model,

    allow_dangerous_deserialization=True

)

print("=" * 60)
print("FAISS Vector Database Loaded Successfully")
print("=" * 60)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2287.73it/s]


FAISS Vector Database Loaded Successfully


In [18]:
# ==========================================================
# Load Gemini LLM
# ==========================================================

%pip install -q langchain-google-genai

try:
    from langchain_google_genai import ChatGoogleGenerativeAI
except ModuleNotFoundError as exc:
    raise ImportError(
        "Could not import ChatGoogleGenerativeAI. Install langchain-google-genai in this notebook kernel."
    ) from exc

if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY is missing. Check your .env file.")

model_names = [
    "gemini-3.1-flash-lite",
]

last_error = None
llm = None


for model_name in model_names:
    try:
        llm = ChatGoogleGenerativeAI(
            model=model_name,
            google_api_key=GOOGLE_API_KEY,
            temperature=0.3
        )
        llm.invoke("Say hello in one word")
        print(f"Using model: {model_name}")
        break
    except Exception as exc:
        last_error = exc
        print(f"Model {model_name} failed: {exc}")

if llm is None:
    print("=" * 60)
    print("Gemini model could not be initialized.")
    print("This is usually caused by an invalid API key or exhausted Google AI quota.")
    print("Please check your Google AI Studio billing/quota and try again.")
    print("=" * 60)
else:
    print("=" * 60)
    print("Gemini Model Loaded Successfully")
    print("=" * 60)



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Using model: gemini-3.1-flash-lite
Gemini Model Loaded Successfully


In [19]:
# ==========================================================
# Create RAG Pipeline
# ==========================================================

%pip install -q langchain-core langchain-community

try:
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnableLambda
except ModuleNotFoundError as exc:
    raise ImportError(
        "Could not import the required LangChain core modules for the RAG pipeline."
    ) from exc

prompt = ChatPromptTemplate.from_template(
    """
Use the following context to answer the user's question.
If the answer is not in the context, say that you do not know.

Context:
{context}

Question:
{question}
"""
)


def answer_with_context(inputs):
    query = inputs["query"]
    docs = vector_db.similarity_search(query, k=3)
    context = "\n\n".join(doc.page_content for doc in docs)

    response = llm.invoke(prompt.format(context=context, question=query))

    return {
        "result": response.content if hasattr(response, "content") else str(response),
        "source_documents": docs,
    }


qa_chain = RunnableLambda(answer_with_context)

print("=" * 60)
print("RAG Pipeline Created Successfully")
print("=" * 60)

Note: you may need to restart the kernel to use updated packages.
RAG Pipeline Created Successfully



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
# ==========================================================
# Test Chatbot
# ==========================================================

query = "What is Machine Learning?"

response = qa_chain.invoke(

    {

        "query": query

    }

)

print("=" * 60)
print("Question")
print("=" * 60)

print(query)

print("\n" + "=" * 60)
print("Answer")
print("=" * 60)

print(response["result"])

print("\n" + "=" * 60)
print("Source Documents")
print("=" * 60)

for i, doc in enumerate(response["source_documents"], start=1):

    print(f"\nSource {i}")

    print("Document :", doc.metadata["document"])

    print("Page :", doc.metadata["page"])

    print("-" * 50)

    print(doc.page_content[:300])

print("\n" + "=" * 60)
print("PHASE 5 COMPLETED")
print("=" * 60)

print("""

✓ Gemini Connected

✓ FAISS Retrieved Relevant Chunks

✓ RAG Pipeline Working

Ready For

Phase 6 : Conversational PDF Chatbot

""")

Question
What is Machine Learning?

Answer
[{'type': 'text', 'text': 'I do not know.', 'extras': {'signature': 'EjQKMgERTTIPzd8BYL/Hrp0wn9GHnQuC/S+znKorDDN6fm49ZJzdls9813arxYzeQ7P6lV3M'}}]

Source Documents

Source 1
Document : CN (4).pdf
Page : 1
--------------------------------------------------
COMPUTER NETWORKS LABORATORY ​ ​ ​ ​ ​ (CSDC - 0236) B.Tech IV th Semester (January – June 2025) Submitted by :- Khushal Garg (23103077) Group-G3 Submitted to :- Dr. Samayveer Singh Department of Computer Science & Engineering Dr. B. R. Ambedkar National Institute of Technology Jalandhar -144008, Pu

Source 2
Document : CN (4).pdf
Page : 34
--------------------------------------------------
data = list(input("Enter binary data: ")) print("Hamming Code: ", hamming_code(data)) elif choice == '5': break else: print("Invalid choice. Try again.") if __name__ == "__main__": main() Error Detection and Correction Techniques Error detection and correction are essential techniques in computer ne

Sourc

In [21]:
from google import genai
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemini-robotics-er-2-preview
models/gemini-2.5-computer-use-preview-10-2025
models/an

# PHASE 6 : CONVERSATIONAL PDF CHATBOT
## 🎯 Objective

In this phase we add **conversational memory** to the RAG pipeline:
- Multi-turn dialogue with history context
- Graceful quota / model error handling
- Source document citations with every answer
- Updated Gemini model fallback list (deprecated models removed)

In [23]:
# ===========================================================
# Load FAISS Vector Database
# ===========================================================

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = FAISS.load_local(
    "../vectorstore/faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

print("=" * 60)
print("FAISS Vector DB Loaded")
print("=" * 60)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2145.97it/s]


FAISS Vector DB Loaded


In [24]:
# ===========================================================
# Load Gemini LLM — Updated fallback list, no deprecated models
# ===========================================================

from langchain_google_genai import ChatGoogleGenerativeAI

# Removed: gemini-1.5-flash-latest (404), gemini-2.5-flash (404)
MODEL_LIST = [
   "gemini-3.1-flash-lite",
]

llm = None
active_model = None

for model_name in MODEL_LIST:
    try:
        candidate = ChatGoogleGenerativeAI(
            model=model_name,
            google_api_key=GOOGLE_API_KEY,
            temperature=0.3,
            convert_system_message_to_human=True,
        )
        candidate.invoke("Hello in one word.")
        llm = candidate
        active_model = model_name
        break
    except Exception as exc:
        err_str = str(exc)
        if "RESOURCE_EXHAUSTED" in err_str or "429" in err_str:
            print(f"⚠️  {model_name}: Free-tier quota exhausted — try again in 24h.")
        elif "NOT_FOUND" in err_str or "404" in err_str:
            print(f"⚠️  {model_name}: Model deprecated/not found — skipping.")
        elif "API_KEY_INVALID" in err_str or "401" in err_str:
            print("❌ Invalid API key. Get one at https://aistudio.google.com/app/apikey")
            break
        else:
            print(f"⚠️  {model_name} failed: {exc}")

if llm:
    print("=" * 60)
    print(f"✅ Gemini LLM ready  →  {active_model}")
    print("=" * 60)
else:
    print("=" * 60)
    print("❌ All models failed. Check your API key and quota.")
    print("   https://aistudio.google.com/app/apikey")
    print("=" * 60)

✅ Gemini LLM ready  →  gemini-3.1-flash-lite


In [25]:
# ===========================================================
# Build Conversational RAG Chain with Memory
# ===========================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

# In-memory conversation history [user_msg, ai_msg, user_msg, ai_msg, ...]
conversation_history = []

SYSTEM_PROMPT = """You are an expert AI assistant that answers questions based on the provided PDF document context.

Guidelines:
- Answer ONLY from the provided context.
- If the answer is not in the context, say: "I don't have enough information in the documents to answer this."
- Be concise, accurate and well-structured.
- Use bullet points or numbered lists where appropriate.
- Reference the document and page number when possible.

Chat History:
{history}

Context from Documents:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(SYSTEM_PROMPT)


def answer_with_memory(inputs):
    query = inputs["query"]

    # Build history string (last 6 exchanges = 12 items)
    history_pairs = []
    history_window = conversation_history[-12:]
    for i in range(0, len(history_window) - 1, 2):
        history_pairs.append(
            f"Human: {history_window[i]}\nAssistant: {history_window[i+1]}"
        )
    history_str = "\n\n".join(history_pairs) if history_pairs else "No prior conversation."

    # Retrieve relevant chunks
    docs = vector_db.similarity_search(query, k=4)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Generate answer
    response = llm.invoke(
        prompt.format(history=history_str, context=context, question=query)
    )
    answer = response.content if hasattr(response, "content") else str(response)

    # Append to memory
    conversation_history.append(query)
    conversation_history.append(answer)

    return {"answer": answer, "sources": docs}


conv_chain = RunnableLambda(answer_with_memory)

print("=" * 60)
print("Conversational RAG Chain Ready")
print("=" * 60)

Conversational RAG Chain Ready


In [26]:
# ===========================================================
# Helper: ask() — pretty-print Q&A with source citations
# ===========================================================

def ask(question):
    """Ask a question and display answer + source citations."""
    print("\n" + "=" * 60)
    print(f"\n🧑 USER: {question}\n")
    try:
        result = conv_chain.invoke({"query": question})
        print(f"🤖 AI: {result['answer']}")
        print("\n📎 Sources:")
        for doc in result["sources"][:2]:
            meta = doc.metadata
            print(f"   • {meta.get('document','?')} | Page {meta.get('page','?')} | Chunk #{meta.get('chunk_id','?')}")
            print(f"     → {doc.page_content[:120]}...")
    except Exception as exc:
        err = str(exc)
        if "RESOURCE_EXHAUSTED" in err or "429" in err:
            print("⏳ Quota exhausted. Wait ~24h or enable billing.")
        elif "API_KEY_INVALID" in err or "401" in err:
            print("❌ Invalid API key.")
        else:
            print(f"❌ Error: {exc}")

In [27]:
# Turn 1 — Initial question
ask("What is the title and purpose of this document?")



🧑 USER: What is the title and purpose of this document?

🤖 AI: [{'type': 'text', 'text': "I don't have enough information in the documents to answer this. The provided text contains student identification details and technical configuration notes, but it does not state a formal title or the overall purpose of the document.", 'extras': {'signature': 'EjQKMgERTTIP7D/LccARTzicXuXPq6iE+rHTnetR54emyn3oLGrhFqtpqnX1CRbmZrlOFQe9'}}]

📎 Sources:
   • CN (4).pdf | Page 14 | Chunk #18
     → 2. Interface Labeling ●​ All network interfaces, including routers and switches, have been labeled with their respective...
   • CN (4).pdf | Page 31 | Chunk #36
     → DR. B.R. AMBEDKAR NATIONAL INSTITUTE OF TECHNOLOGY, JALANDHAR Week 6 NAME: Khushal Garg ROLL NO.: 23103077 SECTION: 4A G...


In [28]:
# Turn 2 — Follow-up (tests conversational memory)
ask("What are the main experiments or practicals listed in it?")



🧑 USER: What are the main experiments or practicals listed in it?

🤖 AI: [{'type': 'text', 'text': "I don't have enough information in the documents to answer this. The provided text only contains student identification details (Name, Roll No., Section, Group, Branch) and week numbers; it does not list any experiments or practicals.", 'extras': {'signature': 'EjQKMgERTTIP+8C66k0MnQWk3tihXyIn3m6iFk1v9bEg2Tr4Eiv5A7IiGiQ89p7i9D+LuBQ8'}}]

📎 Sources:
   • CN (4).pdf | Page 45 | Chunk #54
     → DR. B.R. AMBEDKAR NATIONAL INSTITUTE OF TECHNOLOGY, JALANDHAR Week 8 NAME: Khushal Garg ROLL NO.: 23103077 SECTION: 4A G...
   • CN (4).pdf | Page 37 | Chunk #45
     → DR. B.R. AMBEDKAR NATIONAL INSTITUTE OF TECHNOLOGY, JALANDHAR Week 7 NAME: Khushal Garg ROLL NO.: 23103077 SECTION: 4A G...


In [29]:
# Turn 3 — Context reference (tests memory continuity)
ask("Can you expand on the first one you mentioned?")



🧑 USER: Can you expand on the first one you mentioned?

🤖 AI: [{'type': 'text', 'text': 'Based on the provided documents, the first objective (associated with Week 5) involves creating a network scenario with four departments: IT, HR, Sales, and Finance. The requirements for this practical are as follows:\n\n*   **Network Setup:** Each department must be a different local area network (LAN) connected through a central router. Each department must include at least four end systems and at least one router.\n*   **Addressing and Labeling:** \n    *   Assign static IP addresses to every end device in the network.\n    *   Label every interface with its corresponding IP address.\n*   **Topology and Routing:**\n    *   Use a different topology for each department.\n    *   Configure RIP routing.\n    *   Demonstrate data transmission between each department.\n    *   Explain the overall topology of the network.\n\n(Source: Document, Page 5)', 'extras': {'signature': 'EjQKMgERTTIPAogl2xqZzP

In [30]:
# ===========================================================
# PHASE 6 COMPLETED
# ===========================================================

print("=" * 60)
print("PHASE 6 COMPLETED")
print("=" * 60)
print(f"""
✅ Conversational Memory   Active
✅ Multi-turn Q&A         Working
✅ Source Citations        Shown
✅ API Error Handling      Graceful
✅ Deprecated Models       Removed

Conversation turns stored: {len(conversation_history) // 2}

Ready For
Phase 7 & 8 : Streamlit Deployment + Professional UI
→ Run:  streamlit run app.py
""")

PHASE 6 COMPLETED

✅ Conversational Memory   Active
✅ Multi-turn Q&A         Working
✅ Source Citations        Shown
✅ API Error Handling      Graceful
✅ Deprecated Models       Removed

Conversation turns stored: 3

Ready For
Phase 7 & 8 : Streamlit Deployment + Professional UI
→ Run:  streamlit run app.py

